# Predictive Anayltics: Support Vector Machines with Regression

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
from run_config import PATHS

In [2]:
#TODO: embedding ansehen -> vielleicht austauschen
#TODO: make more time efficient
#TODO: change to thundersvm -> installation umstädlich
#TODO: funktion einbauen zum 
#TODO: vielleicht PCA adden

In [3]:
GRID_SAMPLE = 100_000_000 # for running on a subset to get good parameter and the use the whole set validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "HEXAGON" # HEXAGON
SPATIAL_ENCODING = "latlong" # options: embedding, latlong
MODE = "full" # options: full, sample
TIME_UNIT = "24H" # options: 1H, 4H, 24H
H3_RES = "8" # options 7,8

In [4]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [5]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.svm import LinearSVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3
from joblib import Memory

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder

## Preparations

In [6]:
INPUT = PATHS.train_test_dir

In [7]:
# Paths
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"

In [8]:
MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [9]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [10]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71243,2025-10-31,10,5,0,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71244,2025-10-31,10,5,0,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71245,2025-10-04,10,6,0,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71246,2025-10-04,10,6,0,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [11]:
#print("Currently working on a sample from all the data due to runtime issues")
#train_df = train_df.sample(n=10_000, random_state=42)

In [12]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-08-21,8,4,0,-0.5,-0.866025,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-08-21,8,4,0,-0.5,-0.866025,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-21,8,4,0,-0.5,-0.866025,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-08-21,8,4,0,-0.5,-0.866025,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-08-21,8,4,0,-0.5,-0.866025,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [13]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [14]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [15]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-08-21,8,4,0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71243,2025-10-31,10,5,0,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71244,2025-10-31,10,5,0,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71245,2025-10-04,10,6,0,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71246,2025-10-04,10,6,0,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [16]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: hexa")
    for df in (train_df, val_df, test_df):
        df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

Encoding: latlong and Unit: hexa


In [17]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else: 
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Spatial Embedding

In [18]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "HEXAGON"):

    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])

    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)
        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Features per H3 cell (mean of POI features, train only)
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Required by srai: maps each region to its features
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    feature_cols = feature_cols(train_df)

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


Create y

In [19]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [20]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,15.539008,11,3,...,0,1,0,0,1,0,0,0.030465,-0.743630,0.667897
1,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,11.505268,3,1,...,0,1,0,0,1,0,0,0.030410,-0.744120,0.667353
2,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,14.312547,0,0,...,0,1,0,0,1,0,0,0.030383,-0.746328,0.664884
3,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,11.333030,0,0,...,0,1,0,0,1,0,0,0.029832,-0.743933,0.667589
4,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,15.224873,31,2,...,0,1,0,0,1,0,0,0.029953,-0.743527,0.668035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71243,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,0,6.319861,0,0,...,0,1,0,0,1,0,0,0.029331,-0.744404,0.667085
71244,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,0,12.692406,0,4,...,0,1,0,0,1,0,0,0.030707,-0.744132,0.667326
71245,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,0,15.694878,15,1,...,1,0,0,0,1,0,0,0.029590,-0.743430,0.668159
71246,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,0,13.708801,15,6,...,1,0,0,0,1,0,0,0.029935,-0.743690,0.667854


### Grid Search

In [21]:
model = SVR()

In [22]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,15.539008,11,3,...,0,1,0,0,1,0,0,0.030465,-0.743630,0.667897
1,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,11.505268,3,1,...,0,1,0,0,1,0,0,0.030410,-0.744120,0.667353
2,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,14.312547,0,0,...,0,1,0,0,1,0,0,0.030383,-0.746328,0.664884
3,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,11.333030,0,0,...,0,1,0,0,1,0,0,0.029832,-0.743933,0.667589
4,-0.5,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,15.224873,31,2,...,0,1,0,0,1,0,0,0.029953,-0.743527,0.668035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71243,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,0,6.319861,0,0,...,0,1,0,0,1,0,0,0.029331,-0.744404,0.667085
71244,-1.0,-1.836970e-16,-0.433884,-0.900969,0.0,1.0,0,12.692406,0,4,...,0,1,0,0,1,0,0,0.030707,-0.744132,0.667326
71245,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,0,15.694878,15,1,...,1,0,0,0,1,0,0,0.029590,-0.743430,0.668159
71246,-1.0,-1.836970e-16,-0.974928,-0.222521,0.0,1.0,0,13.708801,15,6,...,1,0,0,0,1,0,0,0.029935,-0.743690,0.667854


In [ ]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVR(max_iter=20_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300, 500],
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 100, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.05}
Best CV score: 0.8134405657457954


### Train Model

In [ ]:
X_val_grid

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,0,15.057051,10,1,...,0,1,0,0,1,0,0,0.030848,-0.746184,0.665025
1,1.000000,6.123234e-17,-0.974928,-0.222521,0.0,1.0,0,15.170750,21,2,...,0,0,1,0,1,0,0,0.031433,-0.745636,0.665612
2,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,0,23.425126,0,2,...,0,1,0,0,1,0,0,0.030856,-0.742823,0.668777
3,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,0,15.170750,21,2,...,0,1,0,0,1,0,0,0.031433,-0.745636,0.665612
4,0.866025,5.000000e-01,-0.974928,-0.222521,0.0,1.0,0,2.272078,0,2,...,0,0,1,0,0,1,0,0.027142,-0.743244,0.668469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11894,0.000000,1.000000e+00,0.000000,1.000000,0.0,1.0,1,14.390740,59,11,...,0,0,1,0,1,0,0,0.031461,-0.744736,0.666617
11895,-0.500000,-8.660254e-01,0.433884,-0.900969,0.0,1.0,0,14.010063,2,2,...,0,0,1,0,0,0,1,0.032168,-0.746399,0.664720
11896,0.000000,1.000000e+00,0.000000,1.000000,0.0,1.0,1,9.237463,0,0,...,0,1,0,0,0,1,0,0.028286,-0.743046,0.668642
11897,0.000000,1.000000e+00,0.000000,1.000000,0.0,1.0,1,15.872004,71,11,...,0,1,0,0,0,1,0,0.029329,-0.743090,0.668549


In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(memo..., tol=0.01))])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](41,)","['month_sin','month_cos','weekday_sin',...,'x','y','z']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,41
regressor_ regressor_: objectFitted regressor.,Pipeline,"Pipeline(memo..., tol=0.01))])"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,StandardScaler,StandardScaler()
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('feature_map', ...), ...]"


In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([  1.98485569,   2.46451877,   1.62550159, ..., -16.58280041,
        15.44511045,  13.4002797 ])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 17.33920434413956
MSE: 4648.612708150557
RMSE: 68.1807356087521
R2 Score: 0.9548798617644968


In [ ]:
# save model
if SPATIAL_UNIT == "HEXAGON":
    dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + H3_RES + "_" + TIME_UNIT + "_svr.joblib")
    dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + H3_RES + "_"  + TIME_UNIT + "_svr.joblib")
else: 
    dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
    dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")